In [10]:
import os 
import sys 
import glob 
import re 
import nltk 
import string 
os.chdir(r'C:\Users\ADMIN\Documents\Nam_3\CS419\project')
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.LSA import LSA
from source.utils.evaluations import Evaluator

pd.set_option('display.max_rows', 1000)
pd.set_option("max_colwidth", 40)

In [11]:
docs_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_docs.csv'
queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_queries.csv'
qrels_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\medline\new_version\new_medline_qrels.csv'

medline_docs_df = pd.read_csv(docs_path)
medline_queries_df = pd.read_csv(queries_path)
medline_qrels_df = pd.read_csv(qrels_path)

docs_df = medline_docs_df[['doc_id', 'doc_text']] 
queries_df = medline_queries_df[['query_id', 'query_text']] 

In [12]:
docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True) 
queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True) 

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23836\4133813531.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  docs_df.rename(columns={'doc_id':'id', 'doc_text':'content'}, inplace=True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23836\4133813531.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  queries_df.rename(columns={'query_id':'id', 'query_text':'content'}, inplace=True)


In [13]:
tp = TextPreprocessing() 
processed_docs = tp.process_dataframe(dataframe=docs_df) 
vocab = tp.get_vocab() 
index = InvertedIndex(processed_docs, vocab)
index._build()

In [14]:
lsa = LSA(processed_docs, index, vocab, dims=200)
lsa._vectorize(use_tfidf=True)
lsa._fit_svd()

In [15]:
queries_records = queries_df.to_dict(orient="records")
qrels = medline_qrels_df.to_dict(orient='records')

In [16]:
def build_qrels_dict(qrels_records):
    qrels_dict = {}

    for row in qrels_records:
        qid = row.get("query_id") or row.get("qid")
        did = row.get("doc_id") or row.get("docid") or row.get("document_id")
        rel = row.get("relevance") or row.get("rel") or row.get("score")

        qrels_dict.setdefault(qid, {})[did] = float(rel)

    return qrels_dict


qrels = build_qrels_dict(qrels)

In [17]:
k = 25

lsa_evaluator = Evaluator(
    queries=queries_records,
    qrels=qrels,
    retriever=lsa,
    normalize_relevance_scores=True,
)

result = lsa_evaluator.evaluate_all(top_k=k, return_results=True, return_df=True)
metrics = result.summary

metrics

{'Precision@25': 0.5864,
 'Recall@25': 0.20247040073448536,
 'F1@25': 0.23651682620522366,
 'MAP@25': 0.5340937123876642,
 'Ndcg@25': 0.4304420999046403}

In [18]:
per_query_df = result.per_query_df
display(per_query_df)

,precision,recall,f1,ap,ndcg,hit,rr,relevant_count,retrieved_count,hit_count,has_relevance
query_id,,,,,,,,,,,
1,0.96,0.585366,0.727273,0.960000,0.715302,1.0,1.000000,41.0,25.0,24.0,1.0
2,0.92,0.353846,0.511111,0.867278,0.664542,1.0,1.000000,65.0,25.0,23.0,1.0
3,0.80,0.155039,0.259740,0.734353,0.421267,1.0,1.000000,129.0,25.0,20.0,1.0
4,0.16,0.200000,0.177778,0.054805,0.242689,1.0,0.500000,20.0,25.0,4.0,1.0
5,0.24,0.375000,0.292683,0.189472,0.418203,1.0,1.000000,16.0,25.0,6.0,1.0
6,0.64,0.275862,0.385542,0.390829,0.509140,1.0,1.000000,58.0,25.0,16.0,1.0
7,0.92,0.294872,0.446602,0.888223,0.671039,1.0,1.000000,78.0,25.0,23.0,1.0
8,0.76,0.166667,0.273381,0.596961,0.414927,1.0,0.500000,114.0,25.0,19.0,1.0
9,1.00,0.304878,0.467290,1.000000,0.646397,1.0,1.000000,82.0,25.0,25.0,1.0
